# Analyse: Gesichtsemotion


In [ ]:
import sys
!{sys.executable} -m pip install transformers torch pillow numpy pandas tqdm


In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from transformers import BeitImageProcessor, BeitForImageClassification

print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())


c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.7.1+cpu
CUDA available: False


In [2]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '05_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_face_emotion.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60

MODEL_NAME = 'Tanneru/Facial-Emotion-Detection-FER-RAFDB-AffectNet-BEIT-Large'
UNKNOWN_LABEL = 'Unbestimmt'

LABEL_MAP = {
    'LABEL_0': 'Angry',
    'LABEL_1': 'Disgust',
    'LABEL_2': 'Fear',
    'LABEL_3': 'Happy',
    'LABEL_4': 'Neutral',
    'LABEL_5': 'Sad',
    'LABEL_6': 'Surprise',
}

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')

df = pd.read_csv(COMBINED_CSV)
if 'influencer_type' not in df.columns and 'source' in df.columns:
    df['influencer_type'] = df['source']


Running on device: cpu


In [3]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)


def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [4]:
processor = BeitImageProcessor.from_pretrained(MODEL_NAME)
model = BeitForImageClassification.from_pretrained(MODEL_NAME).to(device).eval()
id2label = model.config.id2label


def label_to_emotion(label: str) -> str:
    if label is None:
        return UNKNOWN_LABEL
    return LABEL_MAP.get(label, label)


def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]


def predict_emotion(image_path: Path):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[0]

        pred_idx = int(probs.argmax().item())
        raw_label = id2label[pred_idx]
        emotion = label_to_emotion(raw_label)
        conf = float(probs[pred_idx].item())
        return raw_label, emotion, conf
    except Exception:
        return None, None, np.nan


In [5]:
video_emotion_raw = []
video_emotion_name = []
video_emotion_conf = []
video_emotion_unique = []
video_emotion_detected_frames = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    vid = get_video_id(row)
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

    raw_labels = []
    emotions = []
    confs = []

    for fp in frame_paths:
        raw_label, emotion, conf = predict_emotion(fp)
        if raw_label is not None:
            raw_labels.append(raw_label)
            emotions.append(emotion)
            if not np.isnan(conf):
                confs.append(conf)

    if emotions:
        counts = Counter(emotions)
        top_emotion, _ = counts.most_common(1)[0]

        top_raw_candidates = [r for r, e in zip(raw_labels, emotions) if e == top_emotion]
        top_raw = Counter(top_raw_candidates).most_common(1)[0][0] if top_raw_candidates else top_emotion

        top_confs = [c for r, e, c in zip(raw_labels, emotions, confs) if e == top_emotion and not np.isnan(c)]
        top_conf = float(np.mean(top_confs)) if top_confs else (float(np.mean(confs)) if confs else np.nan)

        video_emotion_raw.append(top_raw)
        video_emotion_name.append(top_emotion)
        video_emotion_conf.append(top_conf)
        video_emotion_unique.append(len(counts))
        video_emotion_detected_frames.append(len(emotions))
    else:
        video_emotion_raw.append(UNKNOWN_LABEL)
        video_emotion_name.append(UNKNOWN_LABEL)
        video_emotion_conf.append(np.nan)
        video_emotion_unique.append(np.nan)
        video_emotion_detected_frames.append(0)


Processing videos: 100%|██████████| 500/500 [2:17:50<00:00, 16.54s/it]  


In [6]:
df['emotion_major_beit'] = video_emotion_raw
df['emotion_major_beit_readable'] = video_emotion_name
df['emotion_major_beit_confidence'] = video_emotion_conf
df['emotion_unique_labels'] = video_emotion_unique
df['detected_emotion_frames'] = video_emotion_detected_frames

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved: {OUTPUT_CSV}')
df[['emotion_major_beit', 'emotion_major_beit_readable', 'emotion_major_beit_confidence', 'emotion_unique_labels', 'detected_emotion_frames']].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\05_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_face_emotion.csv


,emotion_major_beit,emotion_major_beit_readable,emotion_major_beit_confidence,emotion_unique_labels,detected_emotion_frames
0,LABEL_6,Surprise,0.642707,1,7
1,LABEL_4,Neutral,0.711005,1,10
2,LABEL_4,Neutral,0.661770,1,8
3,LABEL_6,Surprise,0.464281,2,8
4,LABEL_5,Sad,0.555398,4,8
5,LABEL_4,Neutral,0.529231,2,8
6,LABEL_4,Neutral,0.628246,3,8
7,LABEL_6,Surprise,0.421354,2,8
8,LABEL_6,Surprise,0.750243,1,6
9,LABEL_6,Surprise,0.864405,1,8
